# Multi-Agent Resume Screening System

## Overview

**HR-Agent** is a multi-agent AI system that helps HR teams analyze resumes and select suitable candidates based on job requirements.

The system reads candidate resumes, compares skills with required job skills, calculates a matching score, and provides a final decision.

---

## Agents

### 1. Planner Agent
- Validates candidate data.
- Checks missing inputs.
- Detects prompt injection attempts.

### 2. Resume Analyzer Agent
- Analyzes resume content.
- Matches candidate skills with HR requirements.
- Calculates candidate score.

### 3. Reviewer Agent
- Reviews the score.
- Accepts or rejects candidates based on the decision rule.

---

## Workflow
HR Requirements >> Planner Agent >> Resume Analyzer Agent >> Reviewer Agent >> Candidate Decision

---

## Features

- Multi-agent workflow.
- Resume skill matching.
- Top candidate ranking.
- JSONL file processing.
- Execution logging.

### Security Checks

- Input validation.
- Prompt injection detection.

---

## Dataset

The system uses a JSONL file containing candidate resume information:

- Job title
- Skills
- Experience
- Resume text


In [ ]:
# IMPORTS
# ============================================================

from dataclasses import dataclass, field
from datetime import datetime
from typing import List
import json
# ============================================================
# CANDIDATE PROFILE
# ============================================================

@dataclass
class Candidate:

    name: str
    job_title: str
    resume: str

    score: float = 0
    decision: str = "Pending"

    logs: List[str] = field(default_factory=list)

    def add_log(self, message):

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        log_message = f"[{timestamp}] {message}"

        self.logs.append(log_message)

        print(log_message)

# ============================================================
# HR INPUT
# ============================================================

job_title = input("Enter Job Title: ")

required_skills = input(
    "Enter required skills (comma separated): "
).lower().split(",")
# ============================================================
# BASE AGENT
# ============================================================

class BaseAgent:

    def __init__(self, name):
        self.name = name

    def process(self, candidate):
        raise NotImplementedError

# ============================================================
# Planner Agent
# ============================================================

class PlannerAgent(BaseAgent):

    def process(self, candidate):

        candidate.add_log(f"{self.name}: Starting validation.")

       # Check missing information

        if not candidate.job_title.strip():

           candidate.add_log(
          f"{self.name}: Job title is missing."
    )

           return False


        if not candidate.resume.strip():

            candidate.add_log(
         f"{self.name}: Resume is empty."
    )

            return False

        # Prompt Injection Detection
        blocked_words = [
            "ignore previous instructions",
            "forget your instructions",
            "system prompt"
        ]

        resume_lower = candidate.resume.lower()

        for word in blocked_words:

            if word in resume_lower:

                candidate.add_log(
                    f"{self.name}: Prompt Injection detected."
                )

                return False

        candidate.add_log(f"{self.name}: Validation successful.")

        return True

Enter Job Title: AI eng
Enter required skills (comma separated): ML


In [ ]:
# ============================================================
# Resume Analyzer Agent
# ============================================================
class ResumeAnalyzerAgent(BaseAgent):

    def process(self, candidate):

        candidate.add_log(f"{self.name}: Analyzing resume.")

        resume_text = candidate.resume.lower()

        matched_skills = 0

        for skill in required_skills:

            skill = skill.strip()

            if skill in resume_text:
                matched_skills += 1

        candidate.score = (
            matched_skills / len(required_skills)
        ) * 100

        candidate.add_log(
            f"{self.name}: Matching Score = {candidate.score:.1f}%"
        )

        return True

# ============================================================
# Reviewer Agent
# ============================================================

class ReviewerAgent(BaseAgent):

    def process(self, candidate):

        candidate.add_log(f"{self.name}: Reviewing result.")

        # Reflection (Decision Point)

        if candidate.score >= 70:

            candidate.decision = "Accepted"

            candidate.add_log(
                f"{self.name}: Candidate Accepted."
            )

        else:

            candidate.decision = "Rejected"

            candidate.add_log(
                f"{self.name}: Candidate Rejected."
            )

        return True

In [ ]:
# ============================================================
# HR WORKFLOW
# ============================================================

class HRWorkflow:

    def __init__(self):

        self.agents = [

            PlannerAgent("Planner Agent"),
            ResumeAnalyzerAgent("Resume Analyzer Agent"),
            ReviewerAgent("Reviewer Agent")

        ]

    def run(self, candidate):

        candidate.add_log("=" * 40)
        candidate.add_log("HR WORKFLOW STARTED")
        candidate.add_log("=" * 40)

        for agent in self.agents:

            if not agent.process(candidate):

                candidate.add_log("WORKFLOW TERMINATED")

                return

        candidate.add_log("WORKFLOW COMPLETED")

# ============================================================
# LOAD CANDIDATE FROM JSON
# ============================================================

candidates = []
with open("candidates_extracted_cleand_with_resume.jsonl", "r") as file:

    for line in file:

        candidates.append(json.loads(line))

In [ ]:
print(candidates[0])

{'job_title': 'Data Scientist', 'highest_education_level': 'B.E', 'university': 'Rajiv Gandhi Proudyogiki Vishwavidyalaya', 'relevant_years_of_experience': 0, 'skills': "['Python', 'Statsmodels', 'AWS', 'Machine learning', 'Sklearn', 'Scipy', 'Keras']", 'certifications': '[]', 'category': 'Data Science', 'row_index': 21, 'standard_education': 'Bachelor', 'raw_resume_text': 'Education Details \nMay 2013 to May 2017 B.E   UIT-RGPV\nData Scientist \n\nData Scientist - Matelabs\nSkill Details \nPython- Exprience - Less than 1 year months\nStatsmodels- Exprience - 12 months\nAWS- Exprience - Less than 1 year months\nMachine learning- Exprience - Less than 1 year months\nSklearn- Exprience - Less than 1 year months\nScipy- Exprience - Less than 1 year months\nKeras- Exprience - Less than 1 year monthsCompany Details \ncompany - Matelabs\ndescription - ML Platform for business professionals, dummies and enthusiasts.\n60/A Koramangala 5th block,\nAchievements/Tasks behind sukh sagar, Bengaluru

In [ ]:
# ============================================================
# RUN WORKFLOW
# ============================================================


workflow = HRWorkflow()
results = []
for i, item in enumerate(candidates):

    candidate = Candidate(
        name=f"Candidate {i+1}",
        job_title=job_title,
        resume=item["raw_resume_text"]   # أو المفتاح الصحيح
    )

    workflow.run(candidate)

    results.append({
        "name": candidate.name,
        "score": candidate.score,
        "decision": candidate.decision
    })

[2026-07-09 17:42:31] ========================================
[2026-07-09 17:42:31] HR WORKFLOW STARTED
[2026-07-09 17:42:31] ========================================
[2026-07-09 17:42:31] Planner Agent: Starting validation.
[2026-07-09 17:42:31] Planner Agent: Validation successful.
[2026-07-09 17:42:31] Resume Analyzer Agent: Analyzing resume.
[2026-07-09 17:42:31] Resume Analyzer Agent: Matching Score = 100.0%
[2026-07-09 17:42:31] Reviewer Agent: Reviewing result.
[2026-07-09 17:42:31] Reviewer Agent: Candidate Accepted.
[2026-07-09 17:42:31] WORKFLOW COMPLETED
[2026-07-09 17:42:31] ========================================
[2026-07-09 17:42:31] HR WORKFLOW STARTED
[2026-07-09 17:42:31] ========================================
[2026-07-09 17:42:31] Planner Agent: Starting validation.
[2026-07-09 17:42:31] Planner Agent: Validation successful.
[2026-07-09 17:42:31] Resume Analyzer Agent: Analyzing resume.
[2026-07-09 17:42:31] Resume Analyzer Agent: Matching Score = 0.0%
[2026-07-09

In [ ]:
# ============================================================
# FINAL REPORT
# TOP 5 CANDIDATES
# ============================================================
print("\n========== FINAL REPORT ==========")
print("\n========== TOP 5 CANDIDATES ==========\n")

for i, candidate in enumerate(results[:5], start=1):

    print(f"Rank #{i}")
    print(f"Candidate : {candidate['name']}")
    print(f"Score     : {candidate['score']:.1f}%")
    print(f"Decision  : {candidate['decision']}")
    print("-" * 40)



========== FINAL REPORT ==========

========== TOP 5 CANDIDATES ==========

Rank #1
Candidate : Candidate 1
Score     : 100.0%
Decision  : Accepted
----------------------------------------
Rank #2
Candidate : Candidate 2
Score     : 0.0%
Decision  : Rejected
----------------------------------------
Rank #3
Candidate : Candidate 3
Score     : 100.0%
Decision  : Accepted
----------------------------------------
Rank #4
Candidate : Candidate 4
Score     : 100.0%
Decision  : Accepted
----------------------------------------
Rank #5
Candidate : Candidate 5
Score     : 0.0%
Decision  : Rejected
----------------------------------------
